# 2012 script - Leon Levy 

#### Strategy: 
1. GET all dataset metadata from 2012 existing collection
2. Extract all DOIs and dataset titles. Separate datasets with files from those without
3. Update metadata for existing datasets (Create JSON files with new metadata for each dataset (`for DOI in list_of_dois: PUT api_call --upload metadata_edit.json`))
4. Add files for datasets that still need them. 
5. Compare list of created datasets to full list and determine how many more to make. 
6. Do normal workflow for remaining images & datasets. 

## 1. GET all dataset metadata

In [1]:
import requests
import pandas as pd
import json

In [2]:
base_url = "https://dataverse.harvard.edu"
api_token = "dd038035-97e1-443f-b25e-aba1f535c296"
alias = 'levy_photos_2012'

In [ ]:
#GET items in dataverse collection
headers = {'X-Dataverse-key': api_token,}

url_dataverse = '%s/api/dataverses/%s/contents' % (base_url, alias)
r = requests.get(url_dataverse, headers=headers)

with open('datasets.json', 'w') as f:
    json.dump(r.json(), f)

In [ ]:
j = open('datasets.json')
datasets = json.load(j)

In [ ]:
dois_2012 = []

for i in datasets["data"]: 
    dois_2012.append(i["identifier"])

In [ ]:
def prepend(list, str):
      
    # Using format()
    str += '{0}'
    list = [str.format(i) for i in list]
    return(list)

list = dois_2012
str = 'doi:10.7910/'
full_dois = prepend(list, str)

In [ ]:
full_dois

In [ ]:
#write dois to file
file = open('full_dois_2012.txt', 'w')
for i in full_dois:
    file.writelines(i+'\n')

file.close()

In [ ]:
#get metadata from DOIs
headers = {'X-Dataverse-key': api_token,}

for doi in full_dois:
    url_datasets = '%s/api/datasets/:persistentId/?persistentId=%s' % (base_url, doi)
    r = requests.get(url_datasets, headers=headers)
    
    with open('dataset_metadata.json', 'a') as f:
        json.dump(r.json(), f)
        f.write(",")

#### Add [ ] to the beginning and end of the json file or else it won't load
And use jq to "prettify" json into "pretty_dataset_metadata.json"

In [ ]:
e = open('pretty_dataset_metadata.json')
dataset_metadata = json.load(e)

In [ ]:
dataset_metadata[0]['data']['latestVersion']['files']

## 2. Separate datasets from those with files and those without

In [ ]:
#get all files for list of dataset dois
all_dataset_files = []
headers = {'X-Dataverse-key': api_token,}

for doi in full_dois:
    url_request = "%s/api/datasets/:persistentId/versions/:draft/files/?persistentId=%s" % (base_url, doi)
    r = requests.get(url_request, headers=headers)
    all_dataset_files.append(json.loads(r.text))

In [ ]:
#find objects with empty "data" values
for i in all_dataset_files:
    print(i['data']==[])

In [ ]:
full_dois

In [ ]:
#manually created csv file with both lists (hopefully they match up right!)

## 3. Update metadata for existing DOIs

1. Test pyDataverse - can I update metadata if DOI is in the PID column? 
2. 

In [3]:
import os
from pyDataverse.utils import read_csv_as_dicts
csv_datasets_filename = 'test_dataset.csv'
#ds_data = read_csv_as_dicts(csv_datasets_filename)

In [47]:
csv_2_df = pd.read_csv('test_dataset.csv')
csv_2_df = csv_2_df.drop(columns = ['org.dataset_id', 'org.to_update'])
csv_2_df.columns = csv_2_df.columns.str.replace('dv.', '')
csv_2_df.head()

/opt/anaconda3/envs/dataverse/lib/python3.7/site-packages/ipykernel_launcher.py:3: FutureWarning: The default value of regex will change from True to False in a future version.
  This is separate from the ipykernel package so we can avoid doing imports until


,org.doi,termsOfUse,title,author,dsDescription,subject,keyword,language,kindOfData,dateOfDeposit,depositor,distributionDate,productionDate,productionPlace,datasetContact,geographicBoundingBox,geographicCoverage,geographicUnit,grantNumber
0,doi:10.7910/DVN/EXFTAX,Content made available under <a href='https://...,A12_23896.JPG,"[{""authorName"": ""Master, Daniel M."", ""authorAf...","[{""dsDescriptionValue"": ""updated"", ""dsDescript...","[""Arts and Humanities""]","[{""keywordValue"": ""Archaeology""}]","[""English""]","[""Photograph""]",2021-09-01,"Mika, Katherine",2021-09-01,2012,Israel,"[{""datasetContactName"": ""Master, Daniel"", ""dat...","[{""westLongitude"": ""31.659769"", ""eastLongitude...","[{""city"": ""Ashkelon""}]","[""City""]","[{""grantNumberAgency"": ""The Leon Levy Foundati..."
1,doi:10.7910/DVN/NCS5JD,Content made available under <a href='https://...,A12_23895.JPG,"[{""authorName"": ""Master, Daniel M."", ""authorAf...","[{""dsDescriptionValue"": ""updated"", ""dsDescript...","[""Arts and Humanities""]","[{""keywordValue"": ""Archaeology""}]","[""English""]","[""Photograph""]",2021-09-01,"Mika, Katherine",2021-09-01,2012,Israel,"[{""datasetContactName"": ""Master, Daniel"", ""dat...","[{""westLongitude"": ""31.659769"", ""eastLongitude...","[{""city"": ""Ashkelon""}]","[""City""]","[{""grantNumberAgency"": ""The Leon Levy Foundati..."
2,doi:10.7910/DVN/MSPYZ5,Content made available under <a href='https://...,A12_23894.JPG,"[{""authorName"": ""Master, Daniel M."", ""authorAf...","[{""dsDescriptionValue"": ""updated"", ""dsDescript...","[""Arts and Humanities""]","[{""keywordValue"": ""Archaeology""}]","[""English""]","[""Photograph""]",2021-09-01,"Mika, Katherine",2021-09-01,2012,Israel,"[{""datasetContactName"": ""Master, Daniel"", ""dat...","[{""westLongitude"": ""31.659769"", ""eastLongitude...","[{""city"": ""Ashkelon""}]","[""City""]","[{""grantNumberAgency"": ""The Leon Levy Foundati..."
3,doi:10.7910/DVN/M9LJT8,Content made available under <a href='https://...,A12_23893.JPG,"[{""authorName"": ""Master, Daniel M."", ""authorAf...","[{""dsDescriptionValue"": ""updated"", ""dsDescript...","[""Arts and Humanities""]","[{""keywordValue"": ""Archaeology""}]","[""English""]","[""Photograph""]",2021-09-01,"Mika, Katherine",2021-09-01,2012,Israel,"[{""datasetContactName"": ""Master, Daniel"", ""dat...","[{""westLongitude"": ""31.659769"", ""eastLongitude...","[{""city"": ""Ashkelon""}]","[""City""]","[{""grantNumberAgency"": ""The Leon Levy Foundati..."
4,doi:10.7910/DVN/UFQPIF,Content made available under <a href='https://...,A12_23892.JPG,"[{""authorName"": ""Master, Daniel M."", ""authorAf...","[{""dsDescriptionValue"": ""updated"", ""dsDescript...","[""Arts and Humanities""]","[{""keywordValue"": ""Archaeology""}]","[""English""]","[""Photograph""]",2021-09-01,"Mika, Katherine",2021-09-01,2012,Israel,"[{""datasetContactName"": ""Master, Daniel"", ""dat...","[{""westLongitude"": ""31.659769"", ""eastLongitude...","[{""city"": ""Ashkelon""}]","[""City""]","[{""grantNumberAgency"": ""The Leon Levy Foundati..."


#### Create json string from dataframe
Trying to create multiple json files to pass as data argument for metadata update endpoint. 
(doesn't seem to work well) 

In [57]:
json_data_string = csv_2_df.to_json(orient = "index")

In [58]:
json_data_obj = json.loads(json_data_string)

In [92]:
type(json_data_obj)

dict

In [93]:
for idx, item in enumerate(json_data_obj):
    out_file = open('updates/dict_%s.json' % (idx), "w")
    json.dump(item, out_file)
    out_file.close()

In [80]:
for idx, item in enumerate(dicts):
    out_file = open('updates/dict_%s.json' % (idx), "w")
    json.dump(item, out_file)
    out_file.close()

In [4]:
from pyDataverse.models import Dataset
ds_lst = []
for ds in ds_data:
    ds_obj = Dataset()
    ds_obj.set(ds)
    ds_lst.append(ds_obj)

In [5]:
base_url = "https://dataverse.harvard.edu"
api_token = "dd038035-97e1-443f-b25e-aba1f535c296"

from pyDataverse.api import NativeApi
api = NativeApi(base_url, api_token)

In [8]:
ds_data[0]['org.doi']

'doi:10.7910/DVN/EXFTAX'

In [9]:
from pyDataverse.api import NativeApi
api = NativeApi(base_url, api_token)
resp = api.get_info_version()
resp.json()

{'status': 'OK', 'data': {'version': '5.6', 'build': None}}

In [21]:
for doi, ds in zip(ds_data, ds_lst):
    print(doi['\ufefforg.dataset_id'])

1
2
3
4
5


In [30]:
for doi, ds in zip(ds_data, ds_lst):
    out_file = open('updates/%s.json' % (doi['\ufefforg.dataset_id']), "w")
    json.dump(ds.json(), out_file, separators=(',', ':'))
    out_file.close()

In [29]:
#I think the problem is the json files that come from ds.json() :((( -- either create json files directly from CSV or try removing the newlines...
for doi, ds in zip(ds_data, ds_lst):
    resp = api.edit_dataset_metadata(doi['org.doi'], 
                                     open('updates/%s.json' % (doi['\ufefforg.dataset_id'])).read(), 
                                     is_pid=True, 
                                     replace=True, 
                                     auth=True)
    print(resp.status_code)

500
500
500
500
500


In [27]:
for doi, ds in zip(ds_data, ds_lst):
    data = open('updates/%s.json' % (doi['\ufefforg.dataset_id'])).read()
    print(data)

"{\n  \"datasetVersion\": {\n    \"metadataBlocks\": {\n      \"citation\": {\n        \"fields\": [\n          {\n            \"typeName\": \"dateOfDeposit\",\n            \"multiple\": false,\n            \"typeClass\": \"primitive\",\n            \"value\": \"2021-09-01\"\n          },\n          {\n            \"typeName\": \"depositor\",\n            \"multiple\": false,\n            \"typeClass\": \"primitive\",\n            \"value\": \"Mika, Katherine\"\n          },\n          {\n            \"typeName\": \"distributionDate\",\n            \"multiple\": false,\n            \"typeClass\": \"primitive\",\n            \"value\": \"2021-09-01\"\n          },\n          {\n            \"typeName\": \"kindOfData\",\n            \"multiple\": true,\n            \"typeClass\": \"primitive\",\n            \"value\": [\n              \"Photograph\"\n            ]\n          },\n          {\n            \"typeName\": \"language\",\n            \"multiple\": true,\n            \"typeClass

In [19]:
headers = {'X-Dataverse-key': api_token,}

for doi, ds in zip(ds_data, ds_lst):
    #data = ds.json()
    url_update_md = '%s/api/datasets/:persistentId/editMetadata?persistentId=%s&replace=true' % (base_url, doi['org.doi'])
    r = requests.put(url_update_md, data=open('grant_info.json','r').read(), headers=headers)
    print(r.json())

{'status': 'OK', 'data': {'id': 209956, 'datasetId': 4043242, 'datasetPersistentId': 'doi:10.7910/DVN/EXFTAX', 'storageIdentifier': 's3://10.7910/DVN/EXFTAX', 'versionState': 'DRAFT', 'lastUpdateTime': '2021-09-02T16:59:17Z', 'createTime': '2020-08-28T16:21:20Z', 'license': 'NONE', 'fileAccessRequest': False, 'metadataBlocks': {'citation': {'displayName': 'Citation Metadata', 'fields': [{'typeName': 'title', 'multiple': False, 'typeClass': 'primitive', 'value': 'A12_23896.JPG'}, {'typeName': 'author', 'multiple': True, 'typeClass': 'compound', 'value': [{'authorName': {'typeName': 'authorName', 'multiple': False, 'typeClass': 'primitive', 'value': 'Levy, Leon'}, 'authorAffiliation': {'typeName': 'authorAffiliation', 'multiple': False, 'typeClass': 'primitive', 'value': 'Harvard University'}}]}, {'typeName': 'datasetContact', 'multiple': True, 'typeClass': 'compound', 'value': [{'datasetContactName': {'typeName': 'datasetContactName', 'multiple': False, 'typeClass': 'primitive', 'value'

In [15]:
with open('grant_info.json') as f:
    d = json.load(f)

In [28]:
headers = {'X-Dataverse-key': api_token,}

url_update_md = '%s/api/datasets/:persistentId/editMetadata?persistentId=doi:10.7910/DVN/EXFTAX&replace=true' % (base_url)
r = requests.put(url_update_md, data=open('updates/1.json','r').read(), headers=headers)
print(r.json())

{}


In [ ]:
base_url = "https://demo.dataverse.org"
api_token = "dd038035-97e1-443f-b25e-aba1f535c296"

In [ ]:
doi = 'doi:10.70122/FK2/754QGC'
data = d
resp = api.edit_dataset_metadata(doi, data, replace=True, auth=True)
resp.status_code

In [ ]:
data['fields'][0]['value']

In [ ]:
data['fields'][0]['value'] = 'Updated Test1'

In [ ]:
data['fields'][0]['value']

In [ ]:
resp = api.edit_dataset_metadata(doi, data, replace=True, auth=True)
resp.status_code

In [ ]:
with open('test_md_update.json', 'w') as f:
        json.dump(ds_lst[0].json(), f)

#### Create dict from csv and then dump into json file

In [36]:
import csv
import json

In [6]:
filename = "test_dataset.csv"

In [13]:
with open(filename,'r') as data:
    for line in csv.DictReader(data):
        print(line['dv.dsDescription'])

[{"dsDescriptionValue": "updated", "dsDescriptionDate": "2021-05-10"}]
[{"dsDescriptionValue": "updated", "dsDescriptionDate": "2021-05-10"}]
[{"dsDescriptionValue": "updated", "dsDescriptionDate": "2021-05-10"}]
[{"dsDescriptionValue": "updated", "dsDescriptionDate": "2021-05-10"}]
[{"dsDescriptionValue": "updated", "dsDescriptionDate": "2021-05-10"}]


In [14]:
csv_2_df = pd.read_csv('test_dataset.csv')
csv_2_df = csv_2_df.drop(columns = ['org.dataset_id', 'org.to_update'])
csv_2_df.columns = csv_2_df.columns.str.replace('dv.', '')
csv_2_df.head()

/opt/anaconda3/envs/dataverse/lib/python3.7/site-packages/ipykernel_launcher.py:3: FutureWarning: The default value of regex will change from True to False in a future version.
  This is separate from the ipykernel package so we can avoid doing imports until


,org.doi,termsOfUse,title,author,dsDescription,subject,keyword,language,kindOfData,dateOfDeposit,depositor,distributionDate,productionDate,productionPlace,datasetContact,geographicBoundingBox,geographicCoverage,geographicUnit,grantNumber
0,doi:10.7910/DVN/EXFTAX,Content made available under <a href='https://...,A12_23896.JPG,"[{""authorName"": ""Master, Daniel M."", ""authorAf...","[{""dsDescriptionValue"": ""updated"", ""dsDescript...","[""Arts and Humanities""]","[{""keywordValue"": ""Archaeology""}]","[""English""]","[""Photograph""]",2021-09-01,"Mika, Katherine",2021-09-01,2012,Israel,"[{""datasetContactName"": ""Master, Daniel"", ""dat...","[{""westLongitude"": ""31.659769"", ""eastLongitude...","[{""city"": ""Ashkelon""}]","[""City""]","[{""grantNumberAgency"": ""The Leon Levy Foundati..."
1,doi:10.7910/DVN/NCS5JD,Content made available under <a href='https://...,A12_23895.JPG,"[{""authorName"": ""Master, Daniel M."", ""authorAf...","[{""dsDescriptionValue"": ""updated"", ""dsDescript...","[""Arts and Humanities""]","[{""keywordValue"": ""Archaeology""}]","[""English""]","[""Photograph""]",2021-09-01,"Mika, Katherine",2021-09-01,2012,Israel,"[{""datasetContactName"": ""Master, Daniel"", ""dat...","[{""westLongitude"": ""31.659769"", ""eastLongitude...","[{""city"": ""Ashkelon""}]","[""City""]","[{""grantNumberAgency"": ""The Leon Levy Foundati..."
2,doi:10.7910/DVN/MSPYZ5,Content made available under <a href='https://...,A12_23894.JPG,"[{""authorName"": ""Master, Daniel M."", ""authorAf...","[{""dsDescriptionValue"": ""updated"", ""dsDescript...","[""Arts and Humanities""]","[{""keywordValue"": ""Archaeology""}]","[""English""]","[""Photograph""]",2021-09-01,"Mika, Katherine",2021-09-01,2012,Israel,"[{""datasetContactName"": ""Master, Daniel"", ""dat...","[{""westLongitude"": ""31.659769"", ""eastLongitude...","[{""city"": ""Ashkelon""}]","[""City""]","[{""grantNumberAgency"": ""The Leon Levy Foundati..."
3,doi:10.7910/DVN/M9LJT8,Content made available under <a href='https://...,A12_23893.JPG,"[{""authorName"": ""Master, Daniel M."", ""authorAf...","[{""dsDescriptionValue"": ""updated"", ""dsDescript...","[""Arts and Humanities""]","[{""keywordValue"": ""Archaeology""}]","[""English""]","[""Photograph""]",2021-09-01,"Mika, Katherine",2021-09-01,2012,Israel,"[{""datasetContactName"": ""Master, Daniel"", ""dat...","[{""westLongitude"": ""31.659769"", ""eastLongitude...","[{""city"": ""Ashkelon""}]","[""City""]","[{""grantNumberAgency"": ""The Leon Levy Foundati..."
4,doi:10.7910/DVN/UFQPIF,Content made available under <a href='https://...,A12_23892.JPG,"[{""authorName"": ""Master, Daniel M."", ""authorAf...","[{""dsDescriptionValue"": ""updated"", ""dsDescript...","[""Arts and Humanities""]","[{""keywordValue"": ""Archaeology""}]","[""English""]","[""Photograph""]",2021-09-01,"Mika, Katherine",2021-09-01,2012,Israel,"[{""datasetContactName"": ""Master, Daniel"", ""dat...","[{""westLongitude"": ""31.659769"", ""eastLongitude...","[{""city"": ""Ashkelon""}]","[""City""]","[{""grantNumberAgency"": ""The Leon Levy Foundati..."


In [16]:
df_2_dict = csv_2_df.to_dict(orient = "index")

In [26]:
for i in df_2_dict:
    print(i, '->', df_2_dict[i]['org.doi'])

0 -> doi:10.7910/DVN/EXFTAX
1 -> doi:10.7910/DVN/NCS5JD
2 -> doi:10.7910/DVN/MSPYZ5
3 -> doi:10.7910/DVN/M9LJT8
4 -> doi:10.7910/DVN/UFQPIF


In [35]:
for i in df_2_dict:
    print(df_2_dict[i])

{'org.doi': 'doi:10.7910/DVN/EXFTAX', 'termsOfUse': "Content made available under <a href='https://creativecommons.org/licenses/by-nc-nd/4.0/'>CC BY-NC-ND 4.0</a> license. ", 'title': 'A12_23896.JPG', 'author': '[{"authorName": "Master, Daniel M.", "authorAffiliation": "Wheaton College"}, {"authorName": "Stager, Lawrence E.", "authorAffiliation": "Harvard University"}]', 'dsDescription': '[{"dsDescriptionValue": "updated", "dsDescriptionDate": "2021-05-10"}]', 'subject': '["Arts and Humanities"]', 'keyword': '[{"keywordValue": "Archaeology"}]', 'language': '["English"]', 'kindOfData': '["Photograph"]', 'dateOfDeposit': '2021-09-01', 'depositor': 'Mika, Katherine', 'distributionDate': '2021-09-01', 'productionDate': 2012, 'productionPlace': 'Israel', 'datasetContact': '[{"datasetContactName": "Master, Daniel", "datasetContactAffiliation": "Wheaton College", "datasetContactEmail": "daniel.master@wheaton.edu"}]', 'geographicBoundingBox': '[{"westLongitude": "31.659769", "eastLongitude": "

In [37]:
df_2_dict[0]

{'org.doi': 'doi:10.7910/DVN/EXFTAX',
 'termsOfUse': "Content made available under <a href='https://creativecommons.org/licenses/by-nc-nd/4.0/'>CC BY-NC-ND 4.0</a> license. ",
 'title': 'A12_23896.JPG',
 'author': '[{"authorName": "Master, Daniel M.", "authorAffiliation": "Wheaton College"}, {"authorName": "Stager, Lawrence E.", "authorAffiliation": "Harvard University"}]',
 'dsDescription': '[{"dsDescriptionValue": "updated", "dsDescriptionDate": "2021-05-10"}]',
 'subject': '["Arts and Humanities"]',
 'keyword': '[{"keywordValue": "Archaeology"}]',
 'language': '["English"]',
 'kindOfData': '["Photograph"]',
 'dateOfDeposit': '2021-09-01',
 'depositor': 'Mika, Katherine',
 'distributionDate': '2021-09-01',
 'productionDate': 2012,
 'productionPlace': 'Israel',
 'datasetContact': '[{"datasetContactName": "Master, Daniel", "datasetContactAffiliation": "Wheaton College", "datasetContactEmail": "daniel.master@wheaton.edu"}]',
 'geographicBoundingBox': '[{"westLongitude": "31.659769", "ea

In [41]:
with open('json_test.json', 'a') as f:
    json.dump(df_2_dict[0], f)

In [40]:
#test edit metadata with this json file:
headers = {'X-Dataverse-key': api_token,}

url_update_md = '%s/api/datasets/:persistentId/editMetadata?persistentId=doi:10.7910/DVN/EXFTAX&replace=true' % (base_url)
r = requests.put(url_update_md, data=open('json_test.json','r').read(), headers=headers)
print(r.json())

{}


#### Back to pyDataverse - dicts need to be transformed into json template, not just have correct key name

In [102]:
import os 
from pyDataverse.utils import read_csv_as_dicts
file = "test_dataset.csv"
ds_dicts = read_csv_as_dicts(file)

In [103]:
for i in ds_dicts:
    del i["\ufefforg.dataset_id"]
    del i['org.doi']
    del i['org.to_update']

In [104]:
ds_dicts[0]

{'termsOfUse': "Content made available under <a href='https://creativecommons.org/licenses/by-nc-nd/4.0/'>CC BY-NC-ND 4.0</a> license. ",
 'title': 'A12_23896.JPG',
 'author': [{'authorName': 'Master, Daniel M.',
   'authorAffiliation': 'Wheaton College'},
  {'authorName': 'Stager, Lawrence E.',
   'authorAffiliation': 'Harvard University'}],
 'dsDescription': [{'dsDescriptionValue': 'updated',
   'dsDescriptionDate': '2021-05-10'}],
 'subject': ['Arts and Humanities'],
 'keyword': [{'keywordValue': 'Archaeology'}],
 'language': ['English'],
 'kindOfData': ['Photograph'],
 'dateOfDeposit': '2021-09-01',
 'depositor': 'Mika, Katherine',
 'distributionDate': '2021-09-01',
 'productionDate': '2012',
 'productionPlace': 'Israel',
 'datasetContact': [{'datasetContactName': 'Master, Daniel',
   'datasetContactAffiliation': 'Wheaton College',
   'datasetContactEmail': 'daniel.master@wheaton.edu'}],
 'geographicBoundingBox': [{'westLongitude': '31.659769',
   'eastLongitude': '31.668014',
   '

In [49]:
from pyDataverse.utils import write_json

In [52]:
write_json("json_test.json", ds_dicts[0])

In [61]:
#test edit metadata with this json file:
headers = {'X-Dataverse-key': api_token,}

url_update_md = '%s/api/datasets/:persistentId/editMetadata?persistentId=doi:10.7910/DVN/EXFTAX&replace=true' % (base_url)
r = requests.put(url_update_md, data=open('json_test.json','r').read(), headers=headers)
print(r.json())

{}


In [74]:
from pyDataverse.models import DVObject
from pyDataverse.models import Dataset

In [106]:
ds_obj = DVObject()
ds_obj.set(ds_dicts[0])

AttributeError: 'DVObject' object has no attribute '_internal_attributes'

In [79]:
for key, val in ds_dicts[0].items():
    print(key, val)

﻿org.dataset_id 1
org.doi doi:10.7910/DVN/EXFTAX
org.to_update True
termsOfUse Content made available under <a href='https://creativecommons.org/licenses/by-nc-nd/4.0/'>CC BY-NC-ND 4.0</a> license. 
title A12_23896.JPG
author [{'authorName': 'Master, Daniel M.', 'authorAffiliation': 'Wheaton College'}, {'authorName': 'Stager, Lawrence E.', 'authorAffiliation': 'Harvard University'}]
dsDescription [{'dsDescriptionValue': 'updated', 'dsDescriptionDate': '2021-05-10'}]
subject ['Arts and Humanities']
keyword [{'keywordValue': 'Archaeology'}]
language ['English']
kindOfData ['Photograph']
dateOfDeposit 2021-09-01
depositor Mika, Katherine
distributionDate 2021-09-01
productionDate 2012
productionPlace Israel
datasetContact [{'datasetContactName': 'Master, Daniel', 'datasetContactAffiliation': 'Wheaton College', 'datasetContactEmail': 'daniel.master@wheaton.edu'}]
geographicBoundingBox [{'westLongitude': '31.659769', 'eastLongitude': '31.668014', 'northLongitude': '34.541372', 'southLongitu

In [76]:
dataset = Dataset()
dataset.set(ds_dicts[0])
dataset.json()

'{\n  "datasetVersion": {\n    "metadataBlocks": {\n      "citation": {\n        "fields": [\n          {\n            "typeName": "dateOfDeposit",\n            "multiple": false,\n            "typeClass": "primitive",\n            "value": "2021-09-01"\n          },\n          {\n            "typeName": "depositor",\n            "multiple": false,\n            "typeClass": "primitive",\n            "value": "Mika, Katherine"\n          },\n          {\n            "typeName": "distributionDate",\n            "multiple": false,\n            "typeClass": "primitive",\n            "value": "2021-09-01"\n          },\n          {\n            "typeName": "kindOfData",\n            "multiple": true,\n            "typeClass": "primitive",\n            "value": [\n              "Photograph"\n            ]\n          },\n          {\n            "typeName": "language",\n            "multiple": true,\n            "typeClass": "controlledVocabulary",\n            "value": [\n              "Engl

In [58]:
ds_obj.json()

'{\n  "datasetVersion": {\n    "metadataBlocks": {\n      "citation": {\n        "fields": [\n          {\n            "typeName": "dateOfDeposit",\n            "multiple": false,\n            "typeClass": "primitive",\n            "value": "2021-09-01"\n          },\n          {\n            "typeName": "depositor",\n            "multiple": false,\n            "typeClass": "primitive",\n            "value": "Mika, Katherine"\n          },\n          {\n            "typeName": "distributionDate",\n            "multiple": false,\n            "typeClass": "primitive",\n            "value": "2021-09-01"\n          },\n          {\n            "typeName": "kindOfData",\n            "multiple": true,\n            "typeClass": "primitive",\n            "value": [\n              "Photograph"\n            ]\n          },\n          {\n            "typeName": "language",\n            "multiple": true,\n            "typeClass": "controlledVocabulary",\n            "value": [\n              "Engl

In [60]:
with open('json_test.json', 'a') as f:
    json.dump(ds_obj.json(), f)

In [62]:
#try to create a dataset just to check...
from pyDataverse.api import NativeApi
api = NativeApi(base_url, api_token)

In [63]:
dv_alias = "levy_photos_2012"
resp = api.create_dataset(dv_alias, ds_obj.json())
#dataset_id_2_pid[ds.get()["org.dataset_id"]] = resp.json()["data"]["persistentId"]

Dataset with pid 'doi:10.7910/DVN/I7Q1QW' created.


In [65]:
doi = "doi:10.7910/DVN/EXFTAX"
resp = api.edit_dataset_metadata(doi, ds_obj.json(), is_pid = True, replace = True, auth = True)

### No way to do this in pyDataverse - either hitting a bug or I just don't understand python/

## Double check that all posssible files have been uploaded

In [1]:
import pandas as pd
import json
import requests

In [3]:
api_token = "899c1144-ae22-4e5e-b89d-c73e2993694f"
base_url = "https://dataverse.harvard.edu"
ID = "levy_photos_2012"

headers = {'X-Dataverse-key': api_token,}

In [2]:
with open("2012/2012_dois.txt", "r") as f:
    doi_file = f.readlines()
    f.close()
    
dois = []
for d in doi_file:
    dois.append(d.rstrip())

dois

['doi:10.7910/DVN/RQUTGG',
 'doi:10.7910/DVN/U4B96B',
 'doi:10.7910/DVN/SUJKSQ',
 'doi:10.7910/DVN/0KRANR',
 'doi:10.7910/DVN/SVT1ND',
 'doi:10.7910/DVN/VYTVBS',
 'doi:10.7910/DVN/TYUPIJ',
 'doi:10.7910/DVN/GESSWE',
 'doi:10.7910/DVN/LTWE5K',
 'doi:10.7910/DVN/HHOLY6',
 'doi:10.7910/DVN/VBUIAA',
 'doi:10.7910/DVN/B5PWMZ',
 'doi:10.7910/DVN/8OJJXV',
 'doi:10.7910/DVN/KZOL68',
 'doi:10.7910/DVN/GEPETL',
 'doi:10.7910/DVN/EOKE9C',
 'doi:10.7910/DVN/EK842P',
 'doi:10.7910/DVN/CMZRCT',
 'doi:10.7910/DVN/2IFS2G',
 'doi:10.7910/DVN/4J63MR',
 'doi:10.7910/DVN/JTC5Y3',
 'doi:10.7910/DVN/WWEE4L',
 'doi:10.7910/DVN/AOG2OF',
 'doi:10.7910/DVN/F05RZM',
 'doi:10.7910/DVN/LAFD5X',
 'doi:10.7910/DVN/B0QQNQ',
 'doi:10.7910/DVN/BM11LP',
 'doi:10.7910/DVN/F3GGQN',
 'doi:10.7910/DVN/BXAHXK',
 'doi:10.7910/DVN/9Q8DVD',
 'doi:10.7910/DVN/7G7UXQ',
 'doi:10.7910/DVN/NZMM3O',
 'doi:10.7910/DVN/M2DYOO',
 'doi:10.7910/DVN/DQDVL0',
 'doi:10.7910/DVN/QKGCEY',
 'doi:10.7910/DVN/ORUM9U',
 'doi:10.7910/DVN/9HUCYK',
 

In [5]:
title = []

for d in dois:
    get_dataset_metadata_url = "%s/api/datasets/:persistentId/?persistentId=%s" % (base_url, d)
    r = requests.get(get_dataset_metadata_url, headers=headers)
    if r.json()['status'] == "OK":
        title.append(r.json()['data']['latestVersion']['metadataBlocks']['citation']['fields'][0]['value'])
        continue

dois_zip = list(zip(dois, title))
doi_df = pd.DataFrame(dois_zip, columns = ['doi', 'title'])

In [6]:
doi_df

,doi,title
0,doi:10.7910/DVN/RQUTGG,A12_20683.JPG
1,doi:10.7910/DVN/U4B96B,A12_20684.JPG
2,doi:10.7910/DVN/SUJKSQ,A12_20685.JPG
3,doi:10.7910/DVN/0KRANR,A12_20686.JPG
4,doi:10.7910/DVN/SVT1ND,A12_20687.JPG
...,...,...
2769,doi:10.7910/DVN/UFQPIF,A12_23892.JPG
2770,doi:10.7910/DVN/M9LJT8,A12_23893.JPG
2771,doi:10.7910/DVN/MSPYZ5,A12_23894.JPG
2772,doi:10.7910/DVN/NCS5JD,A12_23895.JPG


In [14]:
doi_df.to_csv('2012/dois_df.csv')

In [16]:
all_2012 = pd.read_csv("2012/2012_metadata.csv")
uploaded_dois = pd.read_csv("2012/dois_df.csv")

In [18]:
remaining_2012 = all_2012.merge(uploaded_dois, on=['filename'])
result = all_2012[~all_2012.filename.isin(remaining_2012.filename)]
result

,ochre_link,filename
819,http://pi.lib.uchicago.edu/1001/org/ochre/e9e8...,A12_21502JPG
848,http://pi.lib.uchicago.edu/1001/org/ochre/69ca...,A12_21531.jpg
947,http://pi.lib.uchicago.edu/1001/org/ochre/8dc7...,A12_21630.jpg
1288,http://pi.lib.uchicago.edu/1001/org/ochre/c164...,A12_21971.jpg
1785,http://pi.lib.uchicago.edu/1001/org/ochre/6c8f...,A12_22468.jpg
...,...,...
3218,http://pi.lib.uchicago.edu/1001/org/ochre/b8ac...,A12_23903.jpg
3219,http://pi.lib.uchicago.edu/1001/org/ochre/8833...,A12_23904.jpg
3220,http://pi.lib.uchicago.edu/1001/org/ochre/0760...,A12_23905.jpg
3221,http://pi.lib.uchicago.edu/1001/org/ochre/dcd1...,A12_23906.jpg


In [20]:
result['dv.dsDescription'] = '[{"dsDescriptionValue": "Link to OCHRE database: <a href = \'' + result['ochre_link']+ '\'>' + result['ochre_link'] + '\", \"dsDescriptionDate\": \"2022-03-01\"}]'
result.to_csv("missing_data/2012_missing_files.csv") #use this to manually create template files for datasets and datafiles

/opt/anaconda3/envs/dataverse/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  """Entry point for launching an IPython kernel.
